## Sample N images from TCGA folder

In [2]:
import os
import random
import shutil

source_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data"
dest_dir = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\data_cleaning\BG_SVM_training\training_data\tissue"

os.makedirs(dest_dir, exist_ok=True)

# Collect all image paths
image_paths = []

for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
            image_paths.append(os.path.join(root, file))

print(f"Found {len(image_paths)} images")

# Randomly sample 3000
sample_size = min(3000, len(image_paths))
sampled = random.sample(image_paths, sample_size)

# Copy selected images
for img_path in sampled:
    shutil.copy2(img_path, dest_dir)

print(f"Copied {sample_size} images")

Found 1460238 images
Copied 3000 images


## Add the sampled images to csv

In [3]:
import os
import pandas as pd
import numpy as np
from PIL import Image

# ==========================
# ENTER YOUR PATHS HERE
# ==========================

CSV_PATH = r"patch_statistics.csv"

TISSUE_FOLDER = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\data_cleaning\BG_SVM_training\training_data\tissue"
BACKGROUND_FOLDER = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\data_cleaning\BG_SVM_training\training_data\background"

OUTPUT_CSV = r"training_data_statistics.csv"

# ==========================
# FEATURE EXTRACTION
# ==========================

def extract_features(image_path):
    img = Image.open(image_path).convert("RGB")
    img = np.array(img)

    avg_R = img[:, :, 0].mean()
    avg_G = img[:, :, 1].mean()
    avg_B = img[:, :, 2].mean()

    std_R = img[:, :, 0].std()
    std_G = img[:, :, 1].std()
    std_B = img[:, :, 2].std()

    return [
        avg_R,
        avg_G,
        avg_B,
        std_R,
        std_G,
        std_B
    ]


# ==========================
# LOAD EXISTING CSV
# ==========================

df = pd.read_csv(CSV_PATH)

# Existing patch names to avoid duplicates
existing_patches = set(df["patch_name"].astype(str))

new_rows = []

# ==========================
# PROCESS A FOLDER
# ==========================

def process_folder(folder_path, label):

    for file in os.listdir(folder_path):

        if not file.lower().endswith(
            (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
        ):
            continue

        patch_name = file

        if patch_name in existing_patches:
            continue

        image_path = os.path.join(folder_path, file)

        try:
            features = extract_features(image_path)

            new_rows.append([
                patch_name,
                *features,
                label
            ])

        except Exception as e:
            print(f"Error processing {image_path}: {e}")


# ==========================
# ADD TISSUE PATCHES
# ==========================

print("Processing tissue patches...")
process_folder(TISSUE_FOLDER, 0)

# ==========================
# ADD BACKGROUND PATCHES
# ==========================

print("Processing background patches...")
process_folder(BACKGROUND_FOLDER, 1)

# ==========================
# APPEND TO DATAFRAME
# ==========================

new_df = pd.DataFrame(
    new_rows,
    columns=[
        "patch_name",
        "avg_R",
        "avg_G",
        "avg_B",
        "std_R",
        "std_G",
        "std_B",
        "label"
    ]
)

updated_df = pd.concat([df, new_df], ignore_index=True)

# ==========================
# SAVE
# ==========================

updated_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nAdded {len(new_df)} new patches")
print(f"Total rows: {len(updated_df)}")
print(f"Saved to: {OUTPUT_CSV}")

Processing tissue patches...
Processing background patches...

Added 2992 new patches
Total rows: 9410
Saved to: training_data_statistics.csv


## Making background and tissue ratio 70/30

In [11]:
import os
import random
import pandas as pd
import numpy as np
from PIL import Image

# =========================
# PATHS
# =========================

CSV_PATH = r"training_data_statistics.csv"
BACKGROUND_FOLDER = r"D:\Aamir Gulzar\dataset\CRC100K\NCT-CRC-HE-100K-NONORM\BACK"
OUTPUT_CSV = r"training_data_statistics_70_30_new.csv"

# =========================
# FEATURE EXTRACTION
# =========================

def extract_features(image_path):
    img = Image.open(image_path).convert("RGB")
    img = np.array(img)

    return [
        img[:, :, 0].mean(),
        img[:, :, 1].mean(),
        img[:, :, 2].mean(),
        img[:, :, 0].std(),
        img[:, :, 1].std(),
        img[:, :, 2].std()
    ]

# =========================
# LOAD CSV
# =========================

df = pd.read_csv(CSV_PATH)

tissue_count = (df["label"] == 0).sum()
background_count = (df["label"] == 1).sum()

print("Current distribution:")
print("Tissue:", tissue_count)
print("Background:", background_count)

# =========================
# TARGET BACKGROUND (FIXED TISSUE)
# =========================

target_background = int((0.3 / 0.7) * tissue_count)

needed_background = target_background - background_count

print("\nTarget 70/30 split:")
print("Target background:", target_background)
print("Need to add background:", needed_background)

if needed_background <= 0:
    print("Already has enough background. No need to add.")
    exit()

# =========================
# LOAD BACKGROUND IMAGES
# =========================

all_bg_images = []

for root, _, files in os.walk(BACKGROUND_FOLDER):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
            all_bg_images.append(os.path.join(root, f))

print("Total available background images:", len(all_bg_images))

existing = set(df["patch_name"].astype(str))

all_bg_images = [
    p for p in all_bg_images
    if os.path.basename(p) not in existing
]

print("After removing duplicates:", len(all_bg_images))

if len(all_bg_images) < needed_background:
    raise ValueError("Not enough background images available!")

# =========================
# SAMPLE BACKGROUND
# =========================

random.seed(42)
selected_bg = random.sample(all_bg_images, needed_background)

# =========================
# BUILD NEW ROWS
# =========================

new_rows = []

for path in selected_bg:
    try:
        new_rows.append([
            os.path.basename(path),
            *extract_features(path),
            1
        ])
    except Exception as e:
        print("Error:", path, e)

# =========================
# SAVE UPDATED CSV
# =========================

new_df = pd.DataFrame(new_rows, columns=[
    "patch_name",
    "avg_R", "avg_G", "avg_B",
    "std_R", "std_G", "std_B",
    "label"
])

updated_df = pd.concat([df, new_df], ignore_index=True)

updated_df.to_csv(OUTPUT_CSV, index=False)

print("\nDONE")
print("Added background samples:", len(new_df))

print("\nFinal distribution:")
print("Tissue:", (updated_df["label"] == 0).sum())
print("Background:", (updated_df["label"] == 1).sum())

Current distribution:
Tissue: 8067
Background: 1343

Target 70/30 split:
Target background: 3457
Need to add background: 2114
Total available background images: 10566
After removing duplicates: 10566

DONE
Added background samples: 2114

Final distribution:
Tissue: 8067
Background: 3457
